1. **Imports**

In [4]:
# !pip install torch torchvision matplotlib numpy pillow tqdm


import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
# if exists: Gpu
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"used: {DEVICE}")


used: cpu


2. **Hyperparameters**

In [5]:
# input image size 
IMAGE_SIZE = 224

# Μέγεθος grid για YOLO-style detection
# Με ResNet και stride 32, παίρνουμε 224/32 = 7
GRID_SIZE = 7

# Αριθμός bounding boxes ανά cell
NUM_BOXES_PER_CELL = 2

# Batch sze για training
BATCH_SIZE = 16

# Learning rate
LEARNING_RATE = 1e-4

# Αριθμός epochs
NUM_EPOCHS = 10

# Βάρη για τα διάφορα μέρη της loss function
LAMBDA_COORD = 5.0      # Βάρος για localization loss
LAMBDA_NOOBJ = 0.5      # Βάρος για confidence loss όταν δεν υπάρχει αντικείμενο

# Paths για το WIDER FACE dataset
# Άλλαξε αυτά τα paths σύμφωνα με τη δομή του dataset σου
TRAIN_IMAGES_DIR = "WIDER_train/images"
TRAIN_ANNOTATIONS_FILE = "wider_face_split/wider_face_train_bbx_gt.txt"
VAL_IMAGES_DIR = "WIDER_val/images"
VAL_ANNOTATIONS_FILE = "wider_face_split/wider_face_val_bbx_gt.txt"


3. **WIDER FACE annotation file - Parsing**

In [6]:
def parse_wider_face_annotations(annotation_file_path, images_dir):
    """
    Parses annotation file in WIDER FACE dataset.
    
    Returns a list of dicts:
    - 'image_path': image full path
    - 'boxes': list of [x, y, width, height] for every face
    
    annotation file format:
    - Line 1: image name (e.g. "0--Parade/0_Parade_marchingband_1_849.jpg")
    - Line 2: number of faces
    - Lines 3 to 3+n: bounding boxes (x, y, w, h, ...)
    """
    annotations = []
    
    with open(annotation_file_path, 'r') as file:
        lines = file.readlines()
    
    line_index = 0
    while line_index < len(lines):
        # Read image name
        image_name = lines[line_index].strip()
        line_index += 1
        
        # Read number of faces
        num_faces = int(lines[line_index].strip())
        line_index += 1
        
        # Read bounding boxes
        boxes = []
        
        # If no faces exist, line is "0 0 0 0 0 0 0 0 0 0"
        if num_faces == 0:
            line_index += 1  # skip line of zeros
        else:
            for _ in range(num_faces):
                box_data = lines[line_index].strip().split()
                # 1st 4 elements are x, y, width, height
                x = int(box_data[0])
                y = int(box_data[1])
                width = int(box_data[2])
                height = int(box_data[3])
                
                # skip boxes with zero/negative dimensions
                if width > 0 and height > 0:
                    boxes.append([x, y, width, height])
                
                line_index += 1
        
        # An image is appended if it has at least 1 bounding box
        if len(boxes) > 0:
            image_path = os.path.join(images_dir, image_name)
            annotations.append({
                'image_path': image_path,
                'boxes': boxes
            })
    
    return annotations

4. **Dataset Class**

In [7]:
class WiderFaceDataset(Dataset):
    """
    PyTorch Dataset for WIDER FACE dataset.
    
    every image -> to tensor, bounding boxes -> to YOLO-style target tensor.
    """
    
    def __init__(self, annotations, image_size=224, grid_size=7, num_boxes=2):
        """
        Args:
            annotations: dicts list of 'image_path', 'boxes'
            image_size: resizing images
            grid_size: YOLO grid size (7x7)
            num_boxes: bounding boxes per cell
        """
        self.annotations = annotations
        self.image_size = image_size
        self.grid_size = grid_size
        self.num_boxes = num_boxes
        
        # Images Transformatiom
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            # Normalization - ImageNet stats
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, index):
        """
        Returms:
            image: tensor of shape (3, image_size, image_size)
            target: tensor of shape (grid_size, grid_size, 5 * num_boxes)
                    for every cell: [x, y, w, h, confidence] * num_boxes
        """
        annotation = self.annotations[index]
        
        # load image
        image = Image.open(annotation['image_path']).convert('RGB')
        original_width, original_height = image.size
        
        # transform image
        image_tensor = self.transform(image)
        
        # build target tensor
        # Shape: (grid_size, grid_size, 5 * num_boxes)
        # every box has: [x_center, y_center, width, height, confidence]
        target = torch.zeros(self.grid_size, self.grid_size, 5 * self.num_boxes)
        
        # convert every bounding box to YOLO format
        for box in annotation['boxes']:
            x, y, w, h = box
            
            # normalization into [0, 1] for original image
            x_center = (x + w / 2) / original_width
            y_center = (y + h / 2) / original_height
            box_width = w / original_width
            box_height = h / original_height
            
            # in what cell the center of the box belongs to 
            grid_x = int(x_center * self.grid_size)
            grid_y = int(y_center * self.grid_size)
            
            # Clamp so they are inside grid
            grid_x = min(grid_x, self.grid_size - 1)
            grid_y = min(grid_y, self.grid_size - 1)
            
            # compute (relative) co-ordinates inside cell
            # (offset from top-left corner of the cell)
            cell_x = x_center * self.grid_size - grid_x
            cell_y = y_center * self.grid_size - grid_y
            
            # check if cell is complete
            # if not : box in 1st slot
            if target[grid_y, grid_x, 4] == 0:
                target[grid_y, grid_x, 0] = cell_x       # x offset
                target[grid_y, grid_x, 1] = cell_y       # y offset
                target[grid_y, grid_x, 2] = box_width    # width (κανονικοποιημένο)
                target[grid_y, grid_x, 3] = box_height   # height (κανονικοποιημένο)
                target[grid_y, grid_x, 4] = 1.0          # confidence
            # else, in 2nd slot (if exists)
            elif self.num_boxes > 1 and target[grid_y, grid_x, 9] == 0:
                target[grid_y, grid_x, 5] = cell_x
                target[grid_y, grid_x, 6] = cell_y
                target[grid_y, grid_x, 7] = box_width
                target[grid_y, grid_x, 8] = box_height
                target[grid_y, grid_x, 9] = 1.0
        
        return image_tensor, target


5. **ResNet Backbone - YOLO Head**

In [8]:
class FaceDetector(nn.Module):
    """
    Layers:
    1. ResNet18 backbone (frozen) for feature extraction
    2. YOLO-style head for bounding boxes prediction
    
    output with shape: (batch, grid_size, grid_size, 5 * num_boxes)
    each box: [x, y, width, height, confidence]
    """
    
    def __init__(self, grid_size=7, num_boxes=2, freeze_backbone=True):
        super(FaceDetector, self).__init__()
        
        self.grid_size = grid_size
        self.num_boxes = num_boxes
        
        # -----------------------------------
        # 1. ResNet18 Backbone (pretrained)
        # -----------------------------------
        resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        
        # no fully connected layer, no average pooling. keep only convolutional layers
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        
        # freeze weights of the backbone model, if requested
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # number of channels of output for ResNet18 = 512
        backbone_output_channels = 512
        
        # -----------------------------------
        # 2. YOLO-style Detection Head
        # -----------------------------------
        self.detection_head = nn.Sequential(
            # Conv layer 1: less channels
            nn.Conv2d(backbone_output_channels, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            
            # Conv layer 2: more processing
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            
            # Conv layer 3: final output
            # Output channels = 5 * num_boxes (x, y, w, h, conf για κάθε box)
            nn.Conv2d(128, 5 * num_boxes, kernel_size=1),
        )
        
        # Sigmoid for final predictions
        # (x, y, w, h πρέπει να είναι στο [0, 1], confidence επίσης)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        """
        Forward pass.
        Args:
            x: input tensor (shape (batch, 3, 224, 224))
            
        Returns:
            predictions: tensor (shape (batch, grid_size, grid_size, 5 * num_boxes))
        """
        # Feature extraction performed by ResNet backbone
        # Output shape: (batch, 512, 7, 7) for input 224x224
        features = self.backbone(x)
        
        # Detection head
        # Output shape: (batch, 5 * num_boxes, 7, 7)
        predictions = self.detection_head(features)
        
        # sigmoid so that values are inside [0, 1]
        predictions = self.sigmoid(predictions)
        
        # reorganize into (batch, grid_size, grid_size, 5 * num_boxes)
        batch_size = predictions.shape[0]
        predictions = predictions.permute(0, 2, 3, 1).contiguous()
        
        return predictions
    
    def unfreeze_backbone(self):
        """for fine-tuning."""
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("backbone not frozen anymore!")

# building the model...
model = FaceDetector(grid_size=GRID_SIZE, num_boxes=NUM_BOXES_PER_CELL, freeze_backbone=True)
model = model.to(DEVICE)

# print parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total params: {total_params:,}")
print(f"trainable params: {trainable_params:,}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/Androniki/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:06<00:00, 6.90MB/s]

total params: 12,653,514
trainable params: 1,477,002


6. **YOLO Loss Function**

In [9]:
class YOLOLoss(nn.Module):
    """
    YOLO-style loss function for face detection.
    
    loss consists of:
    1. Localization loss: MSE (x, y, w, h)
    2. Confidence loss: MSE -> confidence (different w for obj/noobj)
    """
    
    def __init__(self, grid_size=7, num_boxes=2, lambda_coord=5.0, lambda_noobj=0.5):
        super(YOLOLoss, self).__init__()
        self.grid_size = grid_size
        self.num_boxes = num_boxes
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.mse = nn.MSELoss(reduction='sum')
    
    def forward(self, predictions, targets):
        """        
        Args:
            predictions: (batch, grid, grid, 5 * num_boxes)
            targets: (batch, grid, grid, 5 * num_boxes)
            
        Returns:
            total_loss: scalar tensor
        """
        batch_size = predictions.shape[0]
        
        # initialization
        coord_loss = 0.0
        conf_loss_obj = 0.0
        conf_loss_noobj = 0.0
        
        # for every bounding box predictor
        for b in range(self.num_boxes):
            # Indices for current box
            start_idx = b * 5
            
            # Predictions for this box
            pred_xy = predictions[..., start_idx:start_idx + 2]
            pred_wh = predictions[..., start_idx + 2:start_idx + 4]
            pred_conf = predictions[..., start_idx + 4:start_idx + 5]
            
            # Targets for this box
            target_xy = targets[..., start_idx:start_idx + 2]
            target_wh = targets[..., start_idx + 2:start_idx + 4]
            target_conf = targets[..., start_idx + 4:start_idx + 5]
            
            # Mask: 1 where object exist, 0 elsewhere
            obj_mask = target_conf  # Shape: (batch, grid, grid, 1)
            noobj_mask = 1 - obj_mask
            
            # Localization loss (only for cells with object)
            coord_loss += self.mse(pred_xy * obj_mask, target_xy * obj_mask)
            #sqrt for stability
            coord_loss += self.mse(torch.sqrt(pred_wh + 1e-6) * obj_mask, torch.sqrt(target_wh + 1e-6) * obj_mask)
            
            # Confidence loss for cells with object
            conf_loss_obj += self.mse(pred_conf * obj_mask, target_conf * obj_mask)
            
            # Confidence loss for cells - no object
            conf_loss_noobj += self.mse(pred_conf * noobj_mask, target_conf * noobj_mask)
        
        # total loss. with w
        total_loss = (self.lambda_coord * coord_loss + conf_loss_obj + self.lambda_noobj * conf_loss_noobj)
        
        # normalizing with batch size
        total_loss = total_loss / batch_size
        
        return total_loss

# build loss function...
criterion = YOLOLoss(grid_size=GRID_SIZE, num_boxes=NUM_BOXES_PER_CELL, lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)


7. **Load dataset** 

In [10]:
if os.path.exists(TRAIN_ANNOTATIONS_FILE):
    print("loading annotations...")
    
    # Parse annotations
    train_annotations = parse_wider_face_annotations(TRAIN_ANNOTATIONS_FILE, TRAIN_IMAGES_DIR)
    val_annotations = parse_wider_face_annotations(VAL_ANNOTATIONS_FILE, VAL_IMAGES_DIR)
    
    print(f"Training images: {len(train_annotations)}")
    print(f"Validation images: {len(val_annotations)}")
    
    # creating datasets
    train_dataset = WiderFaceDataset(train_annotations, image_size=IMAGE_SIZE, grid_size=GRID_SIZE, num_boxes=NUM_BOXES_PER_CELL)
    val_dataset = WiderFaceDataset(val_annotations, image_size=IMAGE_SIZE, grid_size=GRID_SIZE, num_boxes=NUM_BOXES_PER_CELL)
    
    # creating dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        
else:
    print("=" * 60)
    print("\nNO files found, pls download dataset here:")
    print("[shuoyang1213.me](http://shuoyang1213.me/WIDERFACE/)")
    print("\n place files in directories:")
    print(f"  - {TRAIN_IMAGES_DIR}/")
    print(f"  - {VAL_IMAGES_DIR}/")
    print(f"  - {TRAIN_ANNOTATIONS_FILE}")
    print(f"  - {VAL_ANNOTATIONS_FILE}")
    print("\nor check file paths")
    
    # dummy data for   testing
    print("\n" + "=" * 60)
    print(" dummy data - testing...")
    print("=" * 60)
    
    class DummyDataset(Dataset):
        def __init__(self, size=100):
            self.size = size
        def __len__(self):
            return self.size
        def __getitem__(self, idx):
            # random
            image = torch.randn(3, IMAGE_SIZE, IMAGE_SIZE)
            # random
            target = torch.zeros(GRID_SIZE, GRID_SIZE, 5 * NUM_BOXES_PER_CELL)
            # into random box
            cx, cy = np.random.randint(0, GRID_SIZE, 2)
            target[cy, cx, 0] = np.random.rand()  # x
            target[cy, cx, 1] = np.random.rand()  # y
            target[cy, cx, 2] = np.random.rand() * 0.5  # w
            target[cy, cx, 3] = np.random.rand() * 0.5  # h
            target[cy, cx, 4] = 1.0  # confidence
            return image, target
    
    train_dataset = DummyDataset(200)
    val_dataset = DummyDataset(50)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


loading annotations...
Training images: 12876
Validation images: 3222


8. **Training Loop**

In [11]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    for 1 epoch    
    Returns:
        average_loss: per epoch
    """
    model.train()
    total_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc="Training")
    for images, targets in progress_bar:
        # transfer to device (cpu/gpu
        images = images.to(device)
        targets = targets.to(device)
        
        # Forward pass
        predictions = model(images)
        loss = criterion(predictions, targets)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # progress bar update
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
    
    average_loss = total_loss / len(train_loader)
    return average_loss


def validate(model, val_loader, criterion, device):
    """    
    Returns:
        average_loss: for validation set
    """
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            
            predictions = model(images)
            loss = criterion(predictions, targets)
            
            total_loss += loss.item()
    
    average_loss = total_loss / len(val_loader)
    return average_loss

9. **training the model...**

In [13]:
# Optimizer - train only detection head (backbone initially frozen)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# for tracking
train_losses = []
val_losses = []
best_val_loss = float('inf')

print("=" * 60)
print("so it begins...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 30)
    
    # Training
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    train_losses.append(train_loss)
    
    # Validation
    val_loss = validate(model, val_loader, criterion, DEVICE)
    val_losses.append(val_loss)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_face_detector.pth')
        print("Νέο καλύτερο μοντέλο αποθηκεύτηκε!")

print("\n" + "=" * 60)
print("training complete!")
print(f"best Validation Loss: {best_val_loss:.4f}")
print("=" * 60)

so it begins...

Epoch 1/10
------------------------------


Training: 100%|██████████| 805/805 [02:33<00:00,  5.23it/s, loss=6.42]


Train Loss: 12.3710
Val Loss: 8.9337
Νέο καλύτερο μοντέλο αποθηκεύτηκε!

Epoch 2/10
------------------------------


Training: 100%|██████████| 805/805 [02:35<00:00,  5.16it/s, loss=10.8]


Train Loss: 8.5308
Val Loss: 8.3148
Νέο καλύτερο μοντέλο αποθηκεύτηκε!

Epoch 3/10
------------------------------


...

Train Loss: 6.1384
Val Loss: 8.2549

Epoch 9/10
------------------------------


Training: 100%|██████████| 805/805 [02:33<00:00,  5.26it/s, loss=7.5] 


Train Loss: 5.8610
Val Loss: 8.3557

Epoch 10/10
------------------------------


Training: 100%|██████████| 805/805 [02:32<00:00,  5.28it/s, loss=4.06]


Train Loss: 5.6759
Val Loss: 8.4796

training complete!
best Validation Loss: 8.0054


10. **Training curves visualization**

In [ ]:

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(val_losses) + 1), val_losses, 'r-', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

print("saved in 'training_curves.png'")


11. **decode predictions**

In [ ]:
def decode_predictions(predictions, confidence_threshold=0.5, grid_size=7):
    """
    turns raw predictions into bounding boxes.
    
    Args:
        predictions: tensor w\ shape (grid_size, grid_size, 5 * num_boxes)
        confidence_threshold: minimum confidence to keep a box
        grid_size: grid size
        
    Returns:
        boxes: list of [x1, y1, x2, y2, confidence]
               to normalized  coordinates [0, 1]
    """
    boxes = []
    num_boxes = predictions.shape[-1] // 5
    
    for i in range(grid_size):
        for j in range(grid_size):
            for b in range(num_boxes):
                start_idx = b * 5
                
                # parameters mining
                x_offset = predictions[i, j, start_idx].item()
                y_offset = predictions[i, j, start_idx + 1].item()
                width = predictions[i, j, start_idx + 2].item()
                height = predictions[i, j, start_idx + 3].item()
                confidence = predictions[i, j, start_idx + 4].item()
                
                if confidence > confidence_threshold:
                    # convert to normalized coordinates
                    x_center = (j + x_offset) / grid_size
                    y_center = (i + y_offset) / grid_size
                    
                    # compute boundries
                    x1 = x_center - width / 2
                    y1 = y_center - height / 2
                    x2 = x_center + width / 2
                    y2 = y_center + height / 2
                    
                    # Clamp in [0, 1]
                    x1 = max(0, min(1, x1))
                    y1 = max(0, min(1, y1))
                    x2 = max(0, min(1, x2))
                    y2 = max(0, min(1, y2))
                    
                    boxes.append([x1, y1, x2, y2, confidence])
    
    return boxes


def visualize_predictions(image_path, model, device, confidence_threshold=0.5):
    """
    shows image w\ predicted bounding boxes.
    
    Args:
        image_path: image path
        model: trained μοντέλο
        device: cpu/gpu
        confidence_threshold: minimum confidence (boxes)
    """
    # load image
    original_image = Image.open(image_path).convert('RGB')
    original_width, original_height = original_image.size
    
    # Transform for model
    transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(
           mean=[0.485, 0.456, 0.406],
           std=[0.229, 0.224, 0.225]
        )
    ])
    
    # Preprocessing
    image_tensor = transform(original_image).unsqueeze(0).to(device)
    
    # Prediction
    model.eval()
    with torch.no_grad():
        predictions = model(image_tensor)
    
    # Decode predictions
    predictions = predictions[0].cpu()  # don't care about batch dimension
    boxes = decode_predictions(predictions, confidence_threshold, GRID_SIZE)
    
    # Visualization
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(original_image)
    
    # draw bounding boxes
    for box in boxes:
        x1, y1, x2, y2, conf = box
        
        # convert to pixel coordinates
        x1_px = x1 * original_width
        y1_px = y1 * original_height
        box_width = (x2 - x1) * original_width
        box_height = (y2 - y1) * original_height
        
        # draw rectangle
        rect = patches.Rectangle(
            (x1_px, y1_px),
            box_width,
            box_height,
            linewidth=2,
            edgecolor='lime',
            facecolor='none'
        )
        ax.add_patch(rect)
        
        # Confidence label
        ax.text(
            x1_px,
            y1_px - 5,
            f'{conf:.2f}',
            color='lime',
            fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.5)
        )
    
    ax.set_title(f'Detected {len(boxes)} face(s)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return boxes
